# 🔗 Prompt Chaining with LCEL

The **prompt chaining** workflow breaks a task into a fixed sequence of simpler
LLM calls, with an optional **quality gate** between stages that stops bad output
from flowing downstream.

This notebook builds the **same email pipeline** as
`01_prompt_chaining_langgraph.ipynb` — same steps,
same validation rule, same two test topics — but assembled from LangChain
Expression Language (LCEL) instead of a `StateGraph`.

```
topic ──▶ extract key points ──▶ [GATE] ──▶ write draft ──▶ polish ──▶ final email
                    ▲               │
                    └── regenerate ─┘  (max 3 attempts, then stop)
```

## Learning Objectives
In this notebook, you will learn:
1. **Sequential composition** - chain steps with the `|` pipe operator instead of `add_edge`
2. **Carrying state forward** - use `RunnablePassthrough.assign` where LangGraph used a `TypedDict`
3. **Quality gates** - implement a validation gate as a runnable that raises an exception
4. **Bounded retries** - replace the `regenerate` loop edge with `.with_retry(stop_after_attempt=3)`
5. **Graceful failure** - replace the `fail_validation` node with `.with_fallbacks(...)`

## Prerequisites
- The gated email chain — `01_prompt_chaining_langgraph.ipynb` (this notebook translates it)
- LCEL basics: `prompt | model | parser` (the LCEL introduction in [repository 03 — LangChain fundamentals](https://github.com/Sourav692/ai-engineering-demystified-03-langchain-fundamentals))
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)

---
## 📦 Step 1: Environment Setup

Load the API key, then create the chat model through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: One model for every step of the chain
# ============================================================================
from aied_helpers import get_openai_llm

# temperature > 0 so a retried extraction produces a *different* sample.
llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print(f"✅ LLM initialized: {llm.model_name} (temperature={llm.temperature})")

---

## 🧱 Part 1: The Three Chain Steps

In LangGraph each step was a **node function** that read from and wrote to a
shared `EmailState` `TypedDict`. In LCEL each step is a small chain:

`prompt template → model → output parser`

`StrOutputParser` pulls the plain text out of the `AIMessage` so the next
template can interpolate it directly.

In [ ]:
# ============================================================================
# CHAIN STEPS: The three LLM calls of the pipeline
# ============================================================================
# Each is prompt -> model -> string. They are composed, not called, here.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

extract_chain = (
    ChatPromptTemplate.from_template("List 3 key topics about: {topic}")
    | llm
    | StrOutputParser()
)

draft_chain = (
    ChatPromptTemplate.from_template(
        "Write a professional email draft covering these points: {key_points}"
    )
    | llm
    | StrOutputParser()
)

polish_chain = (
    ChatPromptTemplate.from_template(
        "Polish this email and add proper greeting/closing: {draft_email}"
    )
    | llm
    | StrOutputParser()
)

print("✅ Three step chains defined")

---

## 🚦 Part 2: The Quality Gate

This is the one place where LCEL and LangGraph genuinely differ.

| | LangGraph | LCEL |
|---|---|---|
| Gate lives in | a check node + a conditional edge function | a `RunnableLambda` in the pipe |
| "Try again" is | an edge back to `extract_points` | an **exception** caught by `.with_retry()` |
| "Give up" is | an edge to a `fail_validation` node | a `.with_fallbacks()` branch |

So the gate does not *return* a route name. It either passes the payload through
untouched, or **raises**. Retrying and giving up are then bolted on by two LCEL
decorators, which is why the gate itself stays this small.

The check is identical to the LangGraph notebook: a temperature-0 judge returns a
structured `GateVerdict` saying whether the key points give a work email a concrete
purpose (a request, decision, deadline or next step).

In [ ]:
# ============================================================================
# QUALITY GATE: Raise when the key points are too vague to draft from
# ============================================================================
from pydantic import BaseModel, Field

MAX_VALIDATION_RETRIES = 3


class GateVerdict(BaseModel):
    """Verdict on the key points extracted in step 1."""

    actionable: bool = Field(
        description=(
            "True only if the key points give a business email a concrete purpose: "
            "a request, decision, deadline or next step for the recipient"
        )
    )
    reason: str = Field(description="One sentence explaining the verdict")


# temperature 0: the gate should judge the same text the same way every time
judge = get_openai_llm(model="gpt-4o-mini").with_structured_output(GateVerdict)


class KeyPointsRejected(Exception):
    """Raised when extracted key points fail the quality gate."""


# Attempt log — reset per run so the printed attempt numbers stay accurate.
attempts: list[str] = []


def validate_key_points(payload: dict) -> dict:
    """Pass the payload through unchanged, or raise to trigger a retry."""
    verdict = judge.invoke(
        "You check step 1 of an email-writing pipeline before step 2 drafts a "
        f"work email from it.\n\nKey points:\n{payload['key_points']}"
    )
    if verdict.actionable:
        print(f"✅ Validation PASSED — {verdict.reason}")
        return payload

    print(
        f"❌ Validation FAILED — {verdict.reason} "
        f"(attempt {len(attempts)}/{MAX_VALIDATION_RETRIES})"
    )
    raise KeyPointsRejected(verdict.reason)


print("✅ Gate defined")

---

## 🔁 Part 3: Extract Plus Validate, with Bounded Retries

`RunnablePassthrough.assign` is the LCEL stand-in for graph state: it runs
`extract_chain` and **adds** the result under `key_points` while keeping `topic`
in the dict, so later steps still see everything.

`.with_retry()` then re-runs *that whole composite* — extract **and** gate —
whenever the gate raises. That is exactly the LangGraph `regenerate` edge, since
retrying the composite re-runs the extraction and gets a fresh sample.

> **Note**: `stop_after_attempt=3` counts total attempts, not retries. The
> extraction runs at most 3 times, matching `MAX_VALIDATION_RETRIES`.

In [ ]:
# ============================================================================
# RETRY WRAPPER: Re-run extraction whenever the gate rejects the output
# ============================================================================
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def extract_with_logging(payload: dict) -> str:
    """Record the attempt, then run the extraction chain."""
    attempts.append(payload["topic"])
    print(f"🔄 Extracting key points (attempt {len(attempts)}/{MAX_VALIDATION_RETRIES})")
    return extract_chain.invoke(payload)


extract_and_validate = (
    RunnablePassthrough.assign(key_points=RunnableLambda(extract_with_logging))
    | RunnableLambda(validate_key_points)
).with_retry(
    retry_if_exception_type=(KeyPointsRejected,),  # only retry gate failures
    stop_after_attempt=MAX_VALIDATION_RETRIES,     # 3 attempts, then re-raise
    wait_exponential_jitter=False,                 # no backoff; the gate is not rate-limited
)

print("✅ Retry-wrapped extraction ready")

---

## 🛑 Part 4: Giving Up Gracefully

When all 3 attempts fail, `.with_retry()` re-raises `KeyPointsRejected`.
`.with_fallbacks()` catches it and runs a replacement runnable — the LCEL
equivalent of LangGraph's `fail_validation` node.

The fallback receives the **original input**, not the exception, so it still has
the topic to report on.

In [ ]:
# ============================================================================
# FALLBACK: Stand in for the fail_validation node when retries are exhausted
# ============================================================================


def fail_validation(payload: dict) -> dict:
    """Produce a terminal 'we stopped' result instead of a weak draft."""
    msg = (
        f"Stopped: could not extract actionable key points after "
        f"{MAX_VALIDATION_RETRIES} attempts. "
        f"Topic may be too abstract: '{payload['topic']}'"
    )
    print(f"🛑 {msg}")
    return {"failed": True, "final_email": msg}


safe_extract = extract_and_validate.with_fallbacks(
    [RunnableLambda(fail_validation)],
    exceptions_to_handle=(KeyPointsRejected,),
)

print("✅ Fallback attached")

---

## 🧩 Part 5: Assembling the Full Pipeline

Two ends have to meet: the happy path continues to draft and polish, the failed
path must skip straight to the result. `RunnableBranch` picks between them by
looking for the `failed` flag the fallback set.

Notice the shape of the happy path — `assign(draft_email=...)` then
`assign(final_email=...)`. Each stage adds one key, so the dict at the end holds
every intermediate value, just like the final `EmailState` did.

In [ ]:
# ============================================================================
# PIPELINE: Wire the gate, the branch and the remaining two steps together
# ============================================================================
from langchain_core.runnables import RunnableBranch

write_and_polish = (
    RunnablePassthrough.assign(draft_email=draft_chain)
    | RunnablePassthrough.assign(final_email=polish_chain)
)

email_pipeline = safe_extract | RunnableBranch(
    # (condition, runnable) pairs first, default last. First match wins.
    (lambda payload: payload.get("failed", False), RunnableLambda(lambda p: p)),
    write_and_polish,  # default: the happy path
)

print("✅ Pipeline assembled")

---

## 🧪 Part 6: Running the Pipeline

Same two test topics as the LangGraph notebook. The first is concrete enough to
clear the gate on the first attempt. The second is abstract, so the gate rejects
it three times and the fallback stops the run instead of emitting a weak email.

In [ ]:
# ============================================================================
# RUNNER: Reset the attempt log, invoke the pipeline, return the final email
# ============================================================================


def create_email(topic: str) -> str:
    """Run the complete email creation pipeline for one topic."""
    attempts.clear()  # fresh attempt numbering per run
    print(f"📧 Creating email about: '{topic}'")
    print("-" * 60)

    result = email_pipeline.invoke({"topic": topic})

    print("\n🎉 Pipeline finished")
    return result["final_email"]

In [ ]:
# ============================================================================
# TEST CASE 1: Concrete topic — should clear the gate and produce an email
# ============================================================================
email_1 = create_email("Request for project deadline extension due to technical challenges")
print(f"\n📄 FINAL EMAIL:\n{email_1}")

In [ ]:
# ============================================================================
# TEST CASE 2: Abstract topic — should fail the gate and stop after 3 attempts
# ============================================================================
email_2 = create_email("The meaning of life and happiness")
print(f"\n📄 FINAL EMAIL:\n{email_2}")

---

## 🔍 Part 7: Inspecting the Chain

Every LCEL chain can describe its own structure. This is the LCEL counterpart to
`compiled_workflow.get_graph().draw_mermaid_png()` in the LangGraph notebook.

`draw_mermaid()` needs no extra packages — paste its output into any Mermaid
renderer. The sibling `print_ascii()` draws directly in the terminal but requires
`grandalf` (add it at the repo root with `uv add --dev grandalf`).

In [ ]:
# ============================================================================
# INTROSPECTION: Print the composed chain's structure as Mermaid source
# ============================================================================
graph = email_pipeline.get_graph()

print("Nodes:")
for node in graph.nodes.values():
    print(f"  - {node.name}")

print("\nMermaid source:\n")
print(graph.draw_mermaid())

# Terminal diagram instead, if you have grandalf installed:
# graph.print_ascii()

---

## 📝 Summary

In this notebook, we rebuilt the LangGraph prompt-chaining email pipeline entirely in LCEL.

### 1. The translation table

| LangGraph | LCEL |
|---|---|
| `StateGraph` plus `TypedDict` state | a dict flowing through a pipe, grown by `RunnablePassthrough.assign` |
| `add_edge(a, b)` | the `\|` pipe operator |
| `add_conditional_edges` back to an earlier node | `.with_retry(stop_after_attempt=N)` |
| a `fail_validation` terminal node | `.with_fallbacks([...])` |
| `add_conditional_edges` to one of two paths | `RunnableBranch` |
| `Annotated[int, operator.add]` retry counter | a plain Python list, reset per run |

### 2. What genuinely differs
- **Retry is exception-driven.** The gate raises rather than returning a route
  name, so "try again" is a decorator on the chain, not an edge in a diagram.
- **No shared mutable state.** Each stage adds a key to the dict passing through.
  There are no reducers because nothing writes concurrently.
- **The loop is bounded by a number, not by a graph.** `stop_after_attempt` is
  the only thing standing between you and an endless regeneration loop.

### 3. When to reach back for LangGraph
If you need to **resume** a half-finished pipeline, **interrupt** it for human
approval, or **checkpoint** intermediate stages, use the LangGraph version. LCEL
chains keep no state between invocations.

### Next Steps
- **`modules/06_routing/notebooks/01_routing_langgraph.ipynb`**: the next workflow pattern — classify the input, then dispatch it to one specialist
- Compare against the LangGraph original in `01_prompt_chaining_langgraph.ipynb`